# Classical Experiment 2 — Photoelectric Effect (Hertz 1887; Lenard 1902; Millikan 1916)
*Background for Weinberg §1.1 ("Photoelectric effect") — the second crack in classical electromagnetism.*

**What the experiments showed (and where classical physics broke)**

- **Hertz (1887):** UV light on metal electrodes makes sparks jump more easily — metals *emit electrons* under light.
- **Lenard (1902), reversing the field:** the ejected electrons have a maximum kinetic energy $K_{\max}$ that
  * does **not** depend on intensity,
  * grows **linearly** with frequency $\nu$,
  * and is **zero below a threshold frequency** $\nu_{\min}$ — *no matter how intense the light or how long you wait*.
- **Classical contradiction:** a wave spreads its energy over the whole surface; a tiny bound electron should need a
  finite accumulation time (est. seconds–hours for weak light) and should be shaken loose by *any* frequency at
  sufficient intensity. Neither is observed: emission is instantaneous, intensity only sets the *number* of electrons.
- **Einstein (1905):** light is quantized in packets $E = h\nu$ (his argument was thermodynamic — entropy of
  radiation obeying Wien's law). One packet transfers all its energy to one electron:
$$K_{\max} = h\nu - W, \qquad W = \text{work function}.$$
- **Millikan (1916):** painstaking $K_{\max}$ vs $\nu$ data for several metals — straight lines, **slope $h$**
  (a constant he set out to disprove), intercept $-W$. The photon is real.


In [ ]:
%matplotlib inline
import numpy as np, matplotlib.pyplot as plt, sys
sys.path.insert(0, '..')
from weinberg_constants import h, e, me, c

eV = 1.602176634e-12                       # erg
metals = [('Na', 2.36), ('Ca', 2.87), ('Zn', 4.31), ('Pt', 5.65)]   # work functions, eV (approx.)
for name, W in metals:
    print(f"{name}:  W = {W:.2f} eV   -> threshold nu_min = {W*eV/h:.3e} Hz"
          f"   (lambda_max = {h*c/(W*eV)*1e7:.0f} nm)")


## 1. Classical prediction vs observation, drawn
**Classical EM** (wave picture): energy flux $\propto$ intensity $I$; electron absorbs until it has enough to escape.
Expect: $K_{\max}$ grows with $I$ (panel a), no threshold, delayed emission. **Observed** (panel b): $K_{\max}$ is a
universal straight line $h\nu - W$; intensity rescales the *current*, not the energy (panel c).

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# (a) classical expectation: K_max ~ absorbed power, any frequency, no threshold
I = np.linspace(0, 10, 200)
axes[0].plot(I, 0.9*np.sqrt(I), lw=2, label='classical: $K_{\\max}$ grows with intensity')
axes[0].plot(I, 0.9*np.sqrt(I)*0.5 + 0.0, lw=2, alpha=0.5)
for nu_lab, cc in [('any $\\nu$ (classical)', 'tab:blue'), ('observed: no ejection below $\\nu_{min}$', 'tab:red')]:
    pass
axes[0].axhline(0, color='tab:red', lw=2, label='observed: $K_{\\max}=0$ below threshold')
axes[0].set_xlabel('intensity $I$'); axes[0].set_ylabel(r'$K_{\max}$ (eV)')
axes[0].set_title('(a) classical wave prediction'); axes[0].legend(fontsize=8, loc='upper left')

# (b) K_max vs nu: straight lines slope h for each metal
nu = np.linspace(0.2e15, 1.8e15, 300)
for name, W in metals:
    K = h*nu/eV - W
    axes[1].plot(nu/1e15, K, lw=2, label=f'{name}, $W$={W:.2f} eV')
    axes[1].plot([W*eV/h/1e15], [0], 'o')
axes[1].axhline(0, color='k', lw=0.8)
axes[1].set_xlabel(r'frequency $\nu$ ($10^{15}$ Hz)'); axes[1].set_ylabel(r'$K_{\max} = h\nu - W$ (eV)')
axes[1].set_title('(b) Einstein photoelectric equation'); axes[1].legend(fontsize=8)

# (c) photocurrent vs retarding voltage: saturation ~ I, same stopping potential V0
V = np.linspace(-1.5, 1.0, 300)
V0 = -0.8                                  # same for all intensities (fixed nu)
for k, I0 in [(1.0, 'high'), (0.6, 'medium'), (0.3, 'low')]:
    cur = k*np.clip(1 - V/V0, 0, None)**1.5
    axes[2].plot(V, cur, lw=2, label=f'intensity {I0}')
axes[2].axvline(V0, color='k', ls='--', lw=1)
axes[2].annotate('$V_0$: independent of intensity', (V0, 0.75), rotation=90, fontsize=8, va='top')
axes[2].set_xlabel('retarding voltage $V$ (V)'); axes[2].set_ylabel('photocurrent')
axes[2].set_title('(c) current scales with $I$, energy does not'); axes[2].legend(fontsize=8)

plt.tight_layout(); plt.show()


## 2. Millikan's test: the slope is Planck's constant
Fit $K_{\max} = h\nu - W$ to mock Millikan-style data for Pt ($W = 5.65$ eV). The slope must come out at $h$ to
within his experimental error ($\sim$0.5 %); the intercept fixes $W$ independently.

In [ ]:
rng = np.random.default_rng(11)
nu_data = np.linspace(1.5e15, 2.5e15, 12)                      # deep UV for Pt
K_data  = h*nu_data/eV - 5.65 + rng.normal(0, 0.04, size=nu_data.size)   # eV, with noise
slope, intercept = np.polyfit(nu_data, K_data, 1)
h_fit = slope*eV

fig, ax = plt.subplots(figsize=(6.5, 4.2))
ax.plot(nu_data/1e15, K_data, 'o', label='data (Pt, schematic)')
ax.plot(nu_data/1e15, np.polyval((slope, intercept), nu_data)/1e0, '-', lw=1.5,
        label=f'fit: slope = {h_fit:.4e} erg s')
ax.axhline(0, color='k', lw=0.8)
ax.set_xlabel(r'$\nu$ ($10^{15}$ Hz)'); ax.set_ylabel(r'$K_{\max}$ (eV)')
ax.set_title('Millikan (1916): slope of $K_{\\max}$ vs $\\nu$ is $h$'); ax.legend()
plt.show()
print(f"h_fit / h = {h_fit/h:.4f}   -> recovers Planck's constant to {(h_fit/h-1)*100:+.2f} %")
print(f"intercept = {intercept:+.3f} eV  (work function W = 5.65 eV)")


## My summary
- Lenard's three facts — instantaneous emission, $K_{\max} \perp I$, $K_{\max} \propto \nu$, and a sharp threshold —
  are all impossible for a classical wave delivering energy smoothly to a point electron.
- Einstein's equation $K_{\max} = h\nu - W$ explains all three with one assumption: light energy comes in quanta
  $h\nu$ that collide one-to-one with electrons. Intensity counts quanta per second; frequency sets quantum size.
- Millikan turned it into precision metrology: the slope of $K_{\max}(\nu)$ *is* Planck's constant, and the intercept
  measures the work function. A property of metal surfaces became a measurement of a universal constant.
- The photon idea survived because it made a quantitative, falsifiable prediction where classical physics had only
  paradoxes.

## Gaps
- Einstein's 1905 argument is thermodynamic (entropy of radiation in the Wien regime $\Rightarrow W = h\nu$):
  reproduce the entropy dilution argument $S' = W'/T'$ symbolically.
- Why is the photoelectric yield strongly frequency-dependent (near threshold) even though only $h\nu$ enters
  $K_{\max}$? (Surface penetration depth / mean free path — outside Weinberg ch. 1.)
- Lenard's "triggering" hypothesis (light only releases pre-existing energy) — what experiment killed it?
  (Intensity-dependence of emission rate without any time lag.)